# Notebook 07 — Advanced Models: LSTM, CNN, DistilBERT (Section 5)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from transformers import AutoTokenizer

from nltk.tokenize import word_tokenize
from collections import Counter
from tqdm import tqdm
import nltk
nltk.download('punkt', quiet=True)

OUTPUTS_DIR = Path("outputs")
MODELS_DIR = Path("models")
LOGS_DIR = Path("logs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load Data

In [2]:
train_df = pd.read_csv(OUTPUTS_DIR / 'train.csv').dropna(subset=['text', 'sentiment'])
test_df = pd.read_csv(OUTPUTS_DIR / 'test.csv').dropna(subset=['text', 'sentiment'])

# Use sentiment task (focus task for advanced models)
le = LabelEncoder()
y_train = le.fit_transform(train_df['sentiment'].astype(str))
y_test = le.transform(test_df['sentiment'].astype(str))
NUM_CLASSES = len(le.classes_)
print(f"Classes: {le.classes_} | Train: {len(train_df)} | Test: {len(test_df)}")

Classes: ['negative' 'neutral' 'positive'] | Train: 19282 | Test: 4821


## 2. Build Vocabulary (for LSTM/CNN)

In [3]:
MAX_VOCAB = 20000
MAX_SEQ_LEN = 100
PAD_IDX = 0
UNK_IDX = 1

def tokenize(text):
    return word_tokenize(str(text).lower())

counter = Counter()
for text in tqdm(train_df['text'], desc="Building vocab"):
    counter.update(tokenize(text))

vocab_words = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(MAX_VOCAB-2)]
word2idx = {w: i for i, w in enumerate(vocab_words)}

def encode(text, max_len=MAX_SEQ_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, UNK_IDX) for t in tokens]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

print(f"Vocab size: {len(word2idx)}")

Building vocab: 100%|██████████| 19282/19282 [00:02<00:00, 7706.76it/s]

Vocab size: 20000


## 3. Dataset

In [4]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.encoded = [encode(t) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.encoded)
    def __getitem__(self, i):
        return torch.tensor(self.encoded[i], dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

train_ds = ReviewDataset(train_df['text'].tolist(), y_train)
test_ds = ReviewDataset(test_df['text'].tolist(), y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

## 4. Bidirectional LSTM

In [5]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        _, (hn, _) = self.lstm(emb)
        # Concatenate final forward and backward hidden states
        out = torch.cat([hn[-2], hn[-1]], dim=1)
        return self.fc(self.dropout(out))


def train_eval(model, train_loader, test_loader, task_name, num_epochs=10, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(log_dir=str(LOGS_DIR / f'{task_name}'))
    
    best_f1 = 0
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(texts), labels)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for texts, labels in test_loader:
                preds_all.extend(model(texts.to(DEVICE)).argmax(1).cpu().numpy())
                labels_all.extend(labels.numpy())
        
        acc = accuracy_score(labels_all, preds_all)
        f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
        writer.add_scalar('Loss/train', total_loss/len(train_loader), epoch)
        writer.add_scalar('F1_macro/test', f1, epoch)
        print(f"[{task_name}] Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f} | F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), MODELS_DIR / f'{task_name}.pt')
    
    writer.close()
    print(f"Best F1: {best_f1:.4f}")
    return best_f1

In [6]:
print("Training BiLSTM...")
lstm_model = BiLSTM(len(word2idx), embed_dim=100, hidden_dim=128, num_classes=NUM_CLASSES).to(DEVICE)
lstm_f1 = train_eval(lstm_model, train_loader, test_loader, 'bilstm_sentiment')

Training BiLSTM...
[bilstm_sentiment] Epoch 1/10 | Loss: 0.7159 | F1: 0.5605
[bilstm_sentiment] Epoch 2/10 | Loss: 0.5843 | F1: 0.5720
[bilstm_sentiment] Epoch 3/10 | Loss: 0.5509 | F1: 0.5817
[bilstm_sentiment] Epoch 4/10 | Loss: 0.5232 | F1: 0.5869
[bilstm_sentiment] Epoch 5/10 | Loss: 0.4979 | F1: 0.6036
[bilstm_sentiment] Epoch 6/10 | Loss: 0.4867 | F1: 0.6053
[bilstm_sentiment] Epoch 7/10 | Loss: 0.4619 | F1: 0.6330
[bilstm_sentiment] Epoch 8/10 | Loss: 0.4421 | F1: 0.6224
[bilstm_sentiment] Epoch 9/10 | Loss: 0.4241 | F1: 0.6399
[bilstm_sentiment] Epoch 10/10 | Loss: 0.4061 | F1: 0.6440
Best F1: 0.6440


## 5. TextCNN

In [7]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, kernel_sizes=(2,3,4), num_filters=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, embed_dim)) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, num_classes)
    
    def forward(self, x):
        emb = self.embedding(x).unsqueeze(1)  # (B, 1, L, E)
        conv_out = [torch.relu(conv(emb)).squeeze(3) for conv in self.convs]
        pooled = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in conv_out]
        cat = torch.cat(pooled, dim=1)
        return self.fc(self.dropout(cat))

import torch.nn.functional as F

print("Training TextCNN...")
cnn_model = TextCNN(len(word2idx), embed_dim=100, num_classes=NUM_CLASSES).to(DEVICE)
cnn_f1 = train_eval(cnn_model, train_loader, test_loader, 'textcnn_sentiment')

Training TextCNN...
[textcnn_sentiment] Epoch 1/10 | Loss: 0.7641 | F1: 0.5763
[textcnn_sentiment] Epoch 2/10 | Loss: 0.6189 | F1: 0.5693
[textcnn_sentiment] Epoch 3/10 | Loss: 0.5696 | F1: 0.5928
[textcnn_sentiment] Epoch 4/10 | Loss: 0.5362 | F1: 0.6174
[textcnn_sentiment] Epoch 5/10 | Loss: 0.5066 | F1: 0.6185
[textcnn_sentiment] Epoch 6/10 | Loss: 0.4691 | F1: 0.6055
[textcnn_sentiment] Epoch 7/10 | Loss: 0.4420 | F1: 0.6354
[textcnn_sentiment] Epoch 8/10 | Loss: 0.4020 | F1: 0.6159
[textcnn_sentiment] Epoch 9/10 | Loss: 0.3602 | F1: 0.6448
[textcnn_sentiment] Epoch 10/10 | Loss: 0.3185 | F1: 0.6216
Best F1: 0.6448


## 6. DistilBERT Fine-tuning

In [8]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class BertDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = labels
    
    def __len__(self): return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Use a subset for faster training
N_TRAIN = min(5000, len(train_df))
N_TEST = min(1000, len(test_df))

bert_train_ds = BertDataset(train_df['text'][:N_TRAIN].tolist(), y_train[:N_TRAIN])
bert_test_ds = BertDataset(test_df['text'][:N_TEST].tolist(), y_test[:N_TEST])
print(f"DistilBERT train: {len(bert_train_ds)} | test: {len(bert_test_ds)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DistilBERT train: 5000 | test: 1000


In [12]:
from sklearn.metrics import f1_score as sk_f1

bert_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': sk_f1(labels, preds, average='macro', zero_division=0)
    }

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / 'distilbert_sentiment'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir=str(LOGS_DIR / 'distilbert_sentiment'),
    logging_steps=50,
    report_to='tensorboard',
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_ds,
    eval_dataset=bert_test_ds,
    compute_metrics=compute_metrics,
)

print("Training DistilBERT...")
trainer.train()
print("Training complete.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training DistilBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.550630,0.536773,0.803000,0.609918
2,0.469143,0.562908,0.795000,0.639350
3,0.342510,0.601179,0.796000,0.660114


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete.


In [15]:
eval_results = trainer.evaluate()
print("DistilBERT eval results:", eval_results)

# Save
trainer.save_model(str(MODELS_DIR / 'distilbert_best'))
tokenizer.save_pretrained(str(MODELS_DIR / 'distilbert_best'))
print("Model saved to models/distilbert_best/")

DistilBERT eval results: {'eval_loss': 0.6852778196334839, 'eval_accuracy': 0.78, 'eval_f1_macro': 0.6238994324279344}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to models/distilbert_best/


## 7. Results Summary

In [16]:
results = pd.DataFrame([
    {'model': 'BiLSTM', 'task': 'sentiment', 'f1_macro': lstm_f1},
    {'model': 'TextCNN', 'task': 'sentiment', 'f1_macro': cnn_f1},
    {'model': 'DistilBERT', 'task': 'sentiment', 'f1_macro': eval_results.get('eval_f1_macro', 0)},
])
print(results)
results.to_csv(OUTPUTS_DIR / 'results_advanced.csv', index=False)
print("TensorBoard: tensorboard --logdir=logs/")

        model       task  f1_macro
0      BiLSTM  sentiment  0.644001
1     TextCNN  sentiment  0.644812
2  DistilBERT  sentiment  0.623899
TensorBoard: tensorboard --logdir=logs/


## Summary
- Bidirectional LSTM (2 layers, 128 hidden)
- TextCNN (kernel sizes 2,3,4; 128 filters each)
- DistilBERT fine-tuned with HuggingFace Trainer
- All trained on sentiment classification task
- Results saved to `outputs/results_advanced.csv`